# Car Damage Classification using Hugging Face Model

# This notebook uses the `beingamit99/car_damage_detection` model from Hugging Face to classify car damage into six categories:
# - Crack
# - Scratch
# - Tire Flat
# - Dent
# - Glass Shatter
# - Lamp Broken


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from dataset import CarDamageDataset
from tqdm import tqdm

In [3]:
# 2. Load Hugging Face Model

from transformers import AutoImageProcessor, AutoModelForImageClassification

model_name = "beingamit99/car_damage_detection"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

# Move model to device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")


/Users/fardinhaque/Downloads/Coding_Projects/DamageDoctor/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Using device: cpu


In [13]:
# 3. Dataset Setup

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
data_dir = 'car_damage_dataset'
train_dir = os.path.join(data_dir, 'training')
val_dir = os.path.join(data_dir, 'validation')

train_dir = "/Users/fardinhaque/Downloads/Coding_Projects/DamageDoctor/HuggingFace/car_damage_dataset/validation/"
val_dir = "/Users/fardinhaque/Downloads/Coding_Projects/DamageDoctor/HuggingFace/car_damage_dataset/training/"

train_dataset = CarDamageDataset(train_dir, transform=transform)
print(type(train_dataset))
print(len(train_dataset))
# print(train_dataset[:10])
val_dataset = CarDamageDataset(val_dir, transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")


<class 'dataset.CarDamageDataset'>
0


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
# 4. Training Function

def train_model(model, train_loader, val_loader, num_epochs=10, device='cpu'):
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    criterion = torch.nn.CrossEntropyLoss()
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader
            
            running_loss = 0.0
            running_corrects = 0
            
            pbar = tqdm(dataloader, desc=f'{phase} batches')
            
            for inputs, labels in pbar:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs).logits
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)
                
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{(torch.sum(preds == labels).item() / inputs.size(0)):.4f}'
                })
            
            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
        
        print()
    
    print('Training complete')
    return model, history


In [ ]:
def create_model():
    # Load pre-trained ResNet50 model
    model = models.resnet50(pretrained=True)
    
    # Freeze all layers except the last few
    for param in list(model.parameters())[:-20]:
        param.requires_grad = False
    
    # Modify the final layer for binary classification
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 1),
        nn.Sigmoid()
    )
    
    return model

In [ ]:
# 5. Train the Model

# Train the model
model, history = train_model(model, train_loader, val_loader, num_epochs=10, device=device)


In [ ]:
# 6. Evaluate the Model

def evaluate_model(model, val_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs).logits
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    damage_types = ['Crack', 'Scratch', 'Tire Flat', 'Dent', 'Glass Shatter', 'Lamp Broken']
    
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=damage_types))
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=damage_types,
                yticklabels=damage_types)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45)
    plt.yticks(rotation=45)
    plt.tight_layout()
    plt.show()

evaluate_model(model, val_loader, device)


In [ ]:
# 7. Save the Model

# Save the model
model.save_pretrained('damage_type')
processor.save_pretrained('damage_type')
print("Model saved successfully.")
